# GALVAT Trial 8: Filter Trial 7 to Roboflow-only checkpoint

This is the small checkpoint workflow only.

It does **not** download new Roboflow datasets and does **not** change the ontology. It restores the Trial 7 training dataset, removes every non-Roboflow image/label pair, audits what remains, reports what classes still need data, and saves a new zip checkpoint to Google Drive.

GALVAT weighted score remains the primary downstream metric; this notebook is only preparing the dataset checkpoint.


In [ ]:
# Cell 1: Setup
!pip -q install pyyaml pandas pillow tqdm matplotlib seaborn

from google.colab import drive
drive.mount('/content/drive')

import datetime
import json
import re
import shutil
import zipfile
from collections import Counter
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import yaml
from tqdm.auto import tqdm

IMAGE_EXTS = {'.jpg', '.jpeg', '.png', '.bmp', '.webp'}
STAMP = datetime.datetime.now().strftime('%Y-%m-%d_%H-%M-%S')


In [ ]:
# Cell 2: Paths and class image goals
# Change SOURCE_ZIP only if your latest Trial 7 dataset zip has a different name.

SOURCE_ZIP = Path('/content/drive/MyDrive/galvat_trial7_training_datasets/galvat_trial7_training_dataset_640_2026-06-11_17-49-31.zip')

WORK_ROOT = Path('/content/galvat_trial8_filter_trial7_to_roboflow_only')
RESTORE_ROOT = WORK_ROOT / 'restore'
WORK_DS = WORK_ROOT / 'dataset'
WORK_META = WORK_ROOT / 'metadata'
DRIVE_OUT = Path('/content/drive/MyDrive/galvat_trial8_dataset_checkpoints')

for p in [RESTORE_ROOT, WORK_META, DRIVE_OUT]:
    p.mkdir(parents=True, exist_ok=True)

# Goals are image-level targets. Edit these later if the target changes.
TARGET_IMAGES = {
    'paper_towel_dispenser': 1500,
    'hearing_aid': 1500,
    'bathroom_sign': 1500,
    'step_curb': 1500,
    'exit_sign': 1500,
    'railing': 1500,
    'urinal': 1500,
    'light_switch': 1500,
    'electrical_outlet': 1500,
    'door': 1500,
    'door_frame': 1500,
    'tupperware': 1000,
    'drinking_glass': 1000,
    'razor': 1000,
    'thermostat': 1000,
    'measuring_cup': 1000,
    'salt_pepper_shaker': 1000,
    'wallet': 1000,
    'soap_dispenser': 1000,
    'pan': 1000,
    'jug': 1000,
    'toothpaste': 1000,
    'writing_device': 1000,
    'medicine_bottle': 1000,
    'toothbrush': 1000,
    'lemon': 1000,
    'trash_can': 1000,
    'deodorant': 1000,
    'keys': 1000,
    'canned_good': 1000,
    'pot': 1000,
    'cane': 1000,
    'coffee_mug': 1000,
    'wine_glass': 1000,
    'eyeglasses': 1000,
    'purse': 1000,
    'toilet': 1000,
    'fork': 1000,
    'knife': 1000,
    'spoon': 1000,
    'cellphone': 1000,
    'tv_remote': 1000,
    'chair': 1000,
    'table': 1000,
    'window': 1000,
}

assert SOURCE_ZIP.exists(), f'Missing SOURCE_ZIP: {SOURCE_ZIP}'
print('Source zip:', SOURCE_ZIP)
print('Work root:', WORK_ROOT)
print('Drive output:', DRIVE_OUT)


In [ ]:
# Cell 3: Unzip Trial 7 dataset to /content and find data.yaml
# This does not modify the Drive zip.

if WORK_DS.exists() and any(WORK_DS.glob('*')):
    print('Working dataset already exists:', WORK_DS)
    print('If you need a clean rerun, delete WORK_ROOT manually and rerun.')
else:
    print('Unzipping:', SOURCE_ZIP)
    with zipfile.ZipFile(SOURCE_ZIP, 'r') as z:
        z.extractall(RESTORE_ROOT)

    data_yaml_candidates = sorted(RESTORE_ROOT.rglob('data.yaml'))
    assert data_yaml_candidates, f'No data.yaml found under {RESTORE_ROOT}'
    BASE_DATA_YAML = data_yaml_candidates[0]
    BASE_DS = BASE_DATA_YAML.parent
    shutil.copytree(BASE_DS, WORK_DS, dirs_exist_ok=True)
    print('Base dataset:', BASE_DS)
    print('Copied working dataset:', WORK_DS)

DATA_YAML = WORK_DS / 'data.yaml'
assert DATA_YAML.exists(), f'Missing working data.yaml: {DATA_YAML}'


In [ ]:
# Cell 4: Load class names and update data.yaml paths for the working copy

def load_yolo_names(data_yaml_path):
    with open(data_yaml_path, 'r') as f:
        data = yaml.safe_load(f)
    names = data.get('names')
    if isinstance(names, dict):
        names = [names[i] if i in names else names[str(i)] for i in range(len(names))]
    return data, list(names)

data_yaml, NAMES = load_yolo_names(DATA_YAML)
data_yaml['path'] = str(WORK_DS)
data_yaml['train'] = 'train/images'
data_yaml['val'] = 'valid/images'
data_yaml['test'] = 'test/images'
data_yaml['names'] = {i: name for i, name in enumerate(NAMES)}
data_yaml['nc'] = len(NAMES)
with open(DATA_YAML, 'w') as f:
    yaml.safe_dump(data_yaml, f, sort_keys=False)

name_to_id = {name: i for i, name in enumerate(NAMES)}
print('Class count:', len(NAMES))
print(NAMES)


In [ ]:
# Cell 5: Remove every non-Roboflow image/label pair
#
# Roboflow examples from our Trial 7 build generally:
# - start with rf_, rf2_, or rf8_
# - OR contain .rf. in the file name
#
# Everything else is treated as non-Roboflow and removed from this checkpoint.

def infer_source_from_name(name):
    s = str(name).lower()
    if s.startswith(('rf_', 'rf2_', 'rf8_')) or '.rf.' in s or '_jpg.rf.' in s or '_png.rf.' in s:
        return 'roboflow'
    if s.startswith('coco_') or '_coco_' in s:
        return 'coco'
    if s.startswith('oi_') or '_oi_' in s:
        return 'openimages'
    return 'unknown_non_roboflow'

removed_rows = []
kept_rows = []

for split in ['train', 'valid', 'test']:
    img_dir = WORK_DS / split / 'images'
    lbl_dir = WORK_DS / split / 'labels'
    img_files = sorted([p for p in img_dir.glob('*') if p.suffix.lower() in IMAGE_EXTS]) if img_dir.exists() else []
    for img_path in tqdm(img_files, desc=f'Filtering {split}'):
        src = infer_source_from_name(img_path.name)
        lbl_path = lbl_dir / f'{img_path.stem}.txt'
        row = {
            'split': split,
            'image': img_path.name,
            'label': lbl_path.name,
            'source': src,
            'had_label_file': lbl_path.exists(),
        }
        if src == 'roboflow':
            kept_rows.append(row)
        else:
            img_path.unlink(missing_ok=True)
            lbl_path.unlink(missing_ok=True)
            removed_rows.append(row)

# Remove orphan labels left behind by any source mismatch.
orphan_rows = []
for split in ['train', 'valid', 'test']:
    img_dir = WORK_DS / split / 'images'
    lbl_dir = WORK_DS / split / 'labels'
    if not lbl_dir.exists():
        continue
    image_stems = {p.stem for p in img_dir.glob('*') if p.suffix.lower() in IMAGE_EXTS}
    for lbl_path in sorted(lbl_dir.glob('*.txt')):
        if lbl_path.stem not in image_stems:
            orphan_rows.append({'split': split, 'label': lbl_path.name})
            lbl_path.unlink(missing_ok=True)

removed_df = pd.DataFrame(removed_rows)
kept_df = pd.DataFrame(kept_rows)
orphan_df = pd.DataFrame(orphan_rows)
removed_df.to_csv(WORK_META / 'removed_non_roboflow_manifest.csv', index=False)
kept_df.to_csv(WORK_META / 'kept_roboflow_manifest.csv', index=False)
orphan_df.to_csv(WORK_META / 'removed_orphan_labels.csv', index=False)

print('Kept Roboflow images:', len(kept_df))
print('Removed non-Roboflow images:', len(removed_df))
print('Removed orphan labels:', len(orphan_df))
print('\nRemoved source breakdown:')
display(removed_df['source'].value_counts().rename_axis('source').reset_index(name='images') if len(removed_df) else pd.DataFrame())
print('\nKept source breakdown:')
display(kept_df['source'].value_counts().rename_axis('source').reset_index(name='images') if len(kept_df) else pd.DataFrame())


In [ ]:
# Cell 6: Audit filtered dataset counts and label validity

def label_file_for_image(img_path):
    return Path(str(img_path).replace('/images/', '/labels/')).with_suffix('.txt')

def audit_dataset():
    split_rows = []
    box_counts = Counter()
    image_counts = Counter()
    empty_counts = Counter()
    bad_rows = []

    for split in ['train', 'valid', 'test']:
        img_dir = WORK_DS / split / 'images'
        img_files = sorted([p for p in img_dir.glob('*') if p.suffix.lower() in IMAGE_EXTS]) if img_dir.exists() else []
        for img_path in img_files:
            lbl_path = label_file_for_image(img_path)
            lines = lbl_path.read_text().splitlines() if lbl_path.exists() else []
            seen = set()
            valid_count = 0
            for line_no, line in enumerate(lines, start=1):
                if not line.strip():
                    continue
                parts = line.split()
                if len(parts) != 5:
                    bad_rows.append({'split': split, 'file': img_path.name, 'line': line_no, 'reason': 'bad_column_count', 'raw': line})
                    continue
                try:
                    cls_id = int(float(parts[0]))
                    x, y, w, h = [float(v) for v in parts[1:]]
                except Exception:
                    bad_rows.append({'split': split, 'file': img_path.name, 'line': line_no, 'reason': 'parse_error', 'raw': line})
                    continue
                if cls_id < 0 or cls_id >= len(NAMES):
                    bad_rows.append({'split': split, 'file': img_path.name, 'line': line_no, 'reason': 'bad_class_id', 'raw': line})
                    continue
                if not (0 <= x <= 1 and 0 <= y <= 1 and 0 < w <= 1 and 0 < h <= 1):
                    bad_rows.append({'split': split, 'file': img_path.name, 'line': line_no, 'reason': 'bad_coords', 'raw': line})
                    continue
                cls = NAMES[cls_id]
                seen.add(cls)
                box_counts[(split, cls)] += 1
                valid_count += 1
            if valid_count == 0:
                empty_counts[split] += 1
            for cls in seen:
                image_counts[(split, cls)] += 1
        split_rows.append({'split': split, 'images': len(img_files), 'empty_label_images': empty_counts[split]})

    class_rows = []
    for cls in NAMES:
        total_images = sum(image_counts[(s, cls)] for s in ['train', 'valid', 'test'])
        total_boxes = sum(box_counts[(s, cls)] for s in ['train', 'valid', 'test'])
        goal = TARGET_IMAGES.get(cls, 1000)
        class_rows.append({
            'class': cls,
            'train_images': image_counts[('train', cls)],
            'valid_images': image_counts[('valid', cls)],
            'test_images': image_counts[('test', cls)],
            'total_images': total_images,
            'train_boxes': box_counts[('train', cls)],
            'valid_boxes': box_counts[('valid', cls)],
            'test_boxes': box_counts[('test', cls)],
            'total_boxes': total_boxes,
            'goal_images': goal,
            'still_need': max(0, goal - total_images),
        })

    split_df = pd.DataFrame(split_rows)
    class_df = pd.DataFrame(class_rows).sort_values(['still_need', 'total_images'], ascending=[False, True])
    bad_df = pd.DataFrame(bad_rows)
    return split_df, class_df, bad_df

split_summary, class_summary, bad_labels = audit_dataset()

split_summary.to_csv(WORK_META / 'roboflow_only_split_summary.csv', index=False)
class_summary.to_csv(WORK_META / 'roboflow_only_class_summary.csv', index=False)
bad_labels.to_csv(WORK_META / 'roboflow_only_bad_labels.csv', index=False)

display(split_summary)
display(class_summary)
print('Bad label rows:', len(bad_labels))
print('Total images:', int(split_summary['images'].sum()))
print('Total labeled boxes:', int(class_summary['total_boxes'].sum()))


In [ ]:
# Cell 7: Classes still needed after filtering

missing = class_summary[class_summary['total_images'] == 0].copy()
needs_more = class_summary[class_summary['still_need'] > 0].copy()

needed_report = {
    'created_at': STAMP,
    'source_zip': str(SOURCE_ZIP),
    'dataset_policy': 'Trial 7 filtered to Roboflow-only images and labels. No new sources imported.',
    'class_count': len(NAMES),
    'total_images_after_filter': int(split_summary['images'].sum()),
    'total_boxes_after_filter': int(class_summary['total_boxes'].sum()),
    'bad_label_rows': int(len(bad_labels)),
    'missing_classes': missing['class'].tolist(),
    'classes_below_goal': needs_more[['class', 'total_images', 'goal_images', 'still_need']].to_dict(orient='records'),
}

(WORK_META / 'roboflow_only_needed_report.json').write_text(json.dumps(needed_report, indent=2))
needs_more[['class', 'total_images', 'goal_images', 'still_need', 'total_boxes']].to_csv(WORK_META / 'roboflow_only_classes_still_needed.csv', index=False)

print('Missing classes:')
display(missing[['class', 'total_images', 'goal_images', 'still_need']])
print('\nClasses below goal:')
display(needs_more[['class', 'total_images', 'goal_images', 'still_need', 'total_boxes']])
print('Saved:', WORK_META / 'roboflow_only_needed_report.json')


In [ ]:
# Cell 8: Simple count chart for review

sns.set_theme(style='whitegrid')
chart_dir = WORK_META / 'charts'
chart_dir.mkdir(exist_ok=True)

plot_df = class_summary.sort_values('total_images', ascending=True).copy()
plt.figure(figsize=(11, max(8, len(plot_df) * 0.22)))
sns.barplot(data=plot_df, y='class', x='total_images', color='steelblue')
for pos, (_, r) in enumerate(plot_df.iterrows()):
    plt.plot([r['goal_images'], r['goal_images']], [pos - 0.35, pos + 0.35], color='firebrick', linewidth=2)
plt.title('Roboflow-only Trial 7 Filter: Images per Class')
plt.xlabel('images containing class; red tick = goal')
plt.ylabel('')
plt.tight_layout()
plt.savefig(chart_dir / 'roboflow_only_class_counts_vs_goal.png', dpi=180)
plt.show()


In [ ]:
# Cell 9: Zip filtered dataset checkpoint to Google Drive

ZIP_LOCAL = Path(f'/content/galvat_trial8_roboflow_only_from_trial7_{STAMP}.zip')
ZIP_DRIVE = DRIVE_OUT / ZIP_LOCAL.name

checkpoint_summary = {
    'created_at': STAMP,
    'source_zip': str(SOURCE_ZIP),
    'work_dataset': str(WORK_DS),
    'metadata_root': str(WORK_META),
    'dataset_policy': 'Filtered Trial 7 dataset to Roboflow-only images and labels.',
    'class_count': len(NAMES),
    'total_images': int(split_summary['images'].sum()),
    'total_boxes': int(class_summary['total_boxes'].sum()),
    'bad_label_rows': int(len(bad_labels)),
    'primary_metric_for_future_training': 'GALVAT weighted score',
    'note': 'No new Roboflow sources were downloaded in this workflow.',
}
(WORK_META / 'roboflow_only_checkpoint_summary.json').write_text(json.dumps(checkpoint_summary, indent=2))

if ZIP_LOCAL.exists():
    ZIP_LOCAL.unlink()

INTERNAL_ROOT = Path('galvat_trial8_roboflow_only_from_trial7')
with zipfile.ZipFile(ZIP_LOCAL, 'w', compression=zipfile.ZIP_DEFLATED, compresslevel=1) as z:
    for path in tqdm(list(WORK_ROOT.rglob('*')), desc='Zipping Roboflow-only checkpoint'):
        if path.is_file():
            z.write(path, INTERNAL_ROOT / path.relative_to(WORK_ROOT))

print('Local zip:', ZIP_LOCAL)
print('Zip size GB:', ZIP_LOCAL.stat().st_size / 1e9)
print('Copying to Drive:', ZIP_DRIVE)
shutil.copy2(ZIP_LOCAL, ZIP_DRIVE)
print('Saved:', ZIP_DRIVE)
print(json.dumps(checkpoint_summary, indent=2))
